# 02 Perfiles Intervalos FUCCI

Reanálisis de los resultados del póster con archivos locales del repositorio. Ejecutar después de instalar requirements.txt. No entrena PINNs. Las salidas nuevas se guardan en results/generated/; las tablas publicadas se conservan.


In [ ]:
OUTPUT_NAME = 'resultados_02_identificabilidad'
from pathlib import Path
import json, hashlib, sys
import os
for key in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS'):
    os.environ.setdefault(key, '1')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, brentq
from scipy.stats import chi2, probplot
from IPython.display import display
try:
    from threadpoolctl import threadpool_limits
    thread_limit = threadpool_limits(limits=1)
except ImportError:
    thread_limit = None  # Instalar requirements.txt para controlar hilos en una sesión ya abierta.
ROOT = Path.cwd().resolve()
if not (ROOT / 'src/common.py').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src/common.py').exists():
    raise RuntimeError('Abra el notebook desde la raíz del repositorio o notebooks/.')
sys.path.insert(0, str(ROOT / 'src'))
from common import load_data, data_arrays, FDSolver, SIGMA, TIMES, BOUNDS
INPUT = ROOT / 'data'
OUT = ROOT / 'results/generated' / OUTPUT_NAME
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size':11, 'figure.dpi':110})
inputs = [INPUT/'datos_reorganizados.csv',
          ROOT/'results/published/real_s1.csv', ROOT/'results/published/real_s2.csv']
manifest = {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs}
import scipy, platform
(OUT/'configuracion.json').write_text(json.dumps(dict(sha256=manifest, sigma=0.05,
    nx=139, inicial='coseno K=8', dominio=[0,1242], python=platform.python_version(),
    numpy=np.__version__, scipy=scipy.__version__), indent=2), encoding='utf-8')


## Modelo compartido con la implementación local
139 nodos; dominio 0–1242 µm; condición inicial coseno K=8; flujo nulo. Se mantienen las cotas locales: difusión 1–2000, tasas 0,0001–0,2. Estas son las cotas de nuestro código, no una reproducción literal de todas las cotas del artículo.

In [ ]:
# El solver se importa de src/common.py para evitar duplicaciones.


## Datos y ajuste clásico desde varios inicios
Se excluye t=0 del objetivo (144 residuos). Se registra éxito de cada optimización.

In [ ]:
df = load_data(INPUT/'datos_reorganizados.csv')
assert len(df)==96 and not df.duplicated(['x','t']).any()
assert np.isfinite(df.to_numpy()).all()
xs, ts, R, G = data_arrays(df)
assert np.allclose(xs, np.arange(24)*54) and np.array_equal(ts,TIMES)
fd = FDSolver(xs,R[0],G[0])
NAMES = {1:['D','kr','kg'],2:['Dr','Dg','kr','kg']}
def unpack(theta,sc):
    return (theta[0],theta[0],theta[1],theta[2]) if sc==1 else tuple(theta)
def scale_for(sc): return np.array([1000 if n.startswith('D') else .05 for n in NAMES[sc]])
def bounds_for(sc):
    return np.array([BOUNDS[n][0] for n in NAMES[sc]]),np.array([BOUNDS[n][1] for n in NAMES[sc]])
def residual(theta,sc,solver=None):
    ro,go=(solver or fd).at_obs(*unpack(theta,sc))
    return np.r_[(ro[1:]-R[1:]).ravel(),(go[1:]-G[1:]).ravel()]/SIGMA

def fit(sc,solver=None):
    scale=scale_for(sc); lo,hi=bounds_for(sc)
    starts=([[700,.03,.07],[300,.05,.1],[1500,.02,.05]] if sc==1 else
            [[600,900,.03,.07],[1100,100,.028,.073],[10,1900,.031,.069],[1500,500,.05,.1]])
    fits=[least_squares(lambda q:residual(q*scale,sc,solver),np.array(p)/scale,
        bounds=(lo/scale,hi/scale),xtol=1e-9,ftol=1e-9,gtol=1e-8,max_nfev=500) for p in starts]
    valid=[a for a in fits if a.success and np.isfinite(a.cost)]
    if not valid: raise RuntimeError('Ningún ajuste convergió: revisar antes de seguir.')
    best=min(valid,key=lambda a:a.cost)
    return best.x*scale,float(2*best.cost),pd.DataFrame([dict(inicio=i,exito=a.success,chi2=2*a.cost,nfev=a.nfev) for i,a in enumerate(fits)])
fits={}
for sc in (1,2):
    theta,cost,log=fit(sc); fits[sc]=(theta,cost)
    log.to_csv(OUT/f'inicios_mle_s{sc}.csv',index=False)
    pd.DataFrame({'parametro':NAMES[sc],'estimacion':theta}).to_csv(OUT/f'mle_s{sc}.csv',index=False)
    print('Escenario',sc,dict(zip(NAMES[sc],theta)),'chi²=',cost); display(log)


## 1 · Perfiles con reoptimización
Se fija un parámetro y se reoptimizan todos los restantes desde varios inicios. No se recortan valores negativos del contraste: si se encuentra un óptimo mejor que la referencia, se detiene el análisis para corregirla. Los perfiles exploran todas las cotas; los extremos sin cruce se etiquetan como limitados por la búsqueda, no como límites de confianza identificados.

In [ ]:
N_GRID = 25  # por parámetro; subir a 41 para mayor resolución gráfica
THRESHOLD = float(chi2.ppf(.95,1))
profiles={}; intervals=[]
def profile_value(sc,j,value):
    theta,base=fits[sc]; scale=scale_for(sc);lo,hi=bounds_for(sc)
    free=[k for k in range(len(theta)) if k!=j]
    qhat=theta/scale
    def fun(v):
        q=qhat.copy();q[j]=value/scale[j];q[free]=v
        return residual(q*scale,sc)
    starts=[qhat[free],(lo[free]+.25*(hi[free]-lo[free]))/scale[free],
            (lo[free]+.75*(hi[free]-lo[free]))/scale[free]]
    attempts=[least_squares(fun,s,bounds=(lo[free]/scale[free],hi[free]/scale[free]),
         xtol=1e-8,ftol=1e-8,gtol=1e-8,max_nfev=400) for s in starts]
    ok=[a for a in attempts if a.success and np.isfinite(a.cost)]
    if not ok: raise RuntimeError(f'Perfil no convergió: escenario {sc}, parámetro {j}, valor {value}')
    best=min(ok,key=lambda a:a.cost);lr=2*best.cost-base
    if lr < -1e-4: raise RuntimeError('Perfil encontró mejor óptimo. Reajustar MLE con este punto antes de continuar.')
    return float(lr)

def endpoint(sc,j,table,side):
    center=fits[sc][0][j]
    part=table[table.valor < center].sort_values('valor',ascending=False) if side=='inferior' else table[table.valor > center].sort_values('valor')
    previous=center
    for row in part.itertuples():
        if row.LR >= THRESHOLD:
            val=brentq(lambda v:profile_value(sc,j,v)-THRESHOLD,min(previous,row.valor),max(previous,row.valor),xtol=1e-7,rtol=1e-7)
            return float(val),'cruce localizado'
        previous=row.valor
    return np.nan,'sin cruce hasta la cota de búsqueda'

for sc in (1,2):
    theta,base=fits[sc];lo,hi=bounds_for(sc)
    for j,name in enumerate(NAMES[sc]):
        # Combina rejilla global, logarítmica para tasas y puntos cerca del óptimo.
        grid=np.linspace(lo[j],hi[j],N_GRID)
        if not name.startswith('D'): grid=np.r_[grid,np.geomspace(lo[j],hi[j],N_GRID)]
        grid=np.unique(np.r_[grid,theta[j],np.clip(theta[j]*np.array([.5,.75,.9,1.1,1.25,1.5]),lo[j],hi[j])])
        rows=[]
        for value in grid:
            lr=profile_value(sc,j,value);rows.append(dict(valor=value,LR=lr))
            pd.DataFrame(rows).to_csv(OUT/f'perfil_s{sc}_{name}.csv',index=False)
        tab=pd.DataFrame(rows);profiles[sc,name]=tab
        low,sl=endpoint(sc,j,tab,'inferior');high,sh=endpoint(sc,j,tab,'superior')
        accepted=tab.LR<=THRESHOLD
        components=int((accepted & ~accepted.shift(fill_value=False)).sum())
        intervals.append(dict(escenario=sc,parametro=name,estimacion=theta[j],inferior=low,superior=high,
            estado_inferior=sl,estado_superior=sh,componentes_en_rejilla=components,cota_inferior=lo[j],cota_superior=hi[j]))
        pd.DataFrame(intervals).to_csv(OUT/'intervalos_perfil.csv',index=False)
        fig,ax=plt.subplots(figsize=(6,4));ax.plot(tab.valor,tab.LR,'o-',ms=3)
        ax.axhline(THRESHOLD,color='darkorange',ls='--',label='Umbral aproximado 95 %')
        ax.axvline(theta[j],color='k',ls=':',label='MLE');ax.set(xlabel=name+(' (µm²/h)' if name.startswith('D') else ' (h⁻¹)'),ylabel='2[ℓ(MLE) − ℓ(perfil)]',title=f'Escenario {sc} · {name}')
        ax.legend();fig.tight_layout();fig.savefig(OUT/f'perfil_s{sc}_{name}.png',dpi=300);plt.show()
        print(sc,name,sl,sh,flush=True)
        if components>1: print('ATENCIÓN: región posiblemente desconectada; los límites mostrados rodean el MLE. Inspeccionar perfil completo.')
display(pd.DataFrame(intervals))


## 2 · Información de Fisher en log-parámetros
La escala logarítmica permite comparar sensibilidades relativas. Se revisa el incremento numérico. Si la matriz es casi singular, no se fabrican intervalos mediante una pseudoinversa. Los intervalos de Wald locales ignoran las cotas; se marcan y se prefieren los perfiles para discutir asimetría.

In [ ]:
fisher_rows=[]
for sc in (1,2):
    theta,_=fits[sc];eta=np.log(theta);k=len(theta)
    def jac(h):
        return np.column_stack([(residual(np.exp(eta+h*np.eye(k)[j]),sc)-residual(np.exp(eta-h*np.eye(k)[j]),sc))/(2*h) for j in range(k)])
    J=jac(1e-4);J2=jac(5e-5);info=J.T@J
    eig=np.linalg.eigvalsh(info);cond=np.linalg.cond(info)
    change=np.linalg.norm(J-J2)/np.linalg.norm(J)
    pd.DataFrame(info,index=NAMES[sc],columns=NAMES[sc]).to_csv(OUT/f'fisher_s{sc}.csv')
    print('Escenario',sc,'autovalores',eig,'condición',cond,'cambio relativo J',change)
    if eig.min()<=0 or cond>1e12 or change>.01:
        print('Fisher numéricamente inestable; no se reportan intervalos locales.');continue
    cov=np.linalg.inv(info);se=np.sqrt(np.diag(cov));corr=cov/np.outer(se,se)
    lo,hi=bounds_for(sc)
    for j,name in enumerate(NAMES[sc]):
        lower,upper=np.exp(eta[j]+np.array([-1,1])*1.95996398454*se[j])
        fisher_rows.append(dict(escenario=sc,parametro=name,inferior_fisher=lower,superior_fisher=upper,excede_cotas=bool(lower<lo[j] or upper>hi[j])))
    pd.DataFrame(corr,index=NAMES[sc],columns=NAMES[sc]).to_csv(OUT/f'correlaciones_s{sc}.csv')
    fig,ax=plt.subplots(figsize=(5,4));im=ax.imshow(corr,vmin=-1,vmax=1,cmap='coolwarm')
    ax.set_xticks(range(k),NAMES[sc]);ax.set_yticks(range(k),NAMES[sc])
    for i in range(k):
        for j in range(k):ax.text(j,i,f'{corr[i,j]:.2f}',ha='center',va='center')
    fig.colorbar(im,ax=ax);ax.set_title(f'Correlaciones locales · escenario {sc}')
    fig.tight_layout();fig.savefig(OUT/f'correlaciones_s{sc}.png',dpi=300);plt.show()
fisher=pd.DataFrame(fisher_rows)
if len(fisher):
    comp=pd.DataFrame(intervals).merge(fisher,on=['escenario','parametro'],how='left')
    comp.to_csv(OUT/'comparacion_intervalos.csv',index=False);display(comp)


## 3 · Comparar escenarios
Se reportan error y AIC sin la constante común de la verosimilitud gaussiana (σ fija). Menor AIC favorece el compromiso ajuste/complejidad, pero no demuestra identificabilidad ni verdad biológica. No se calcula cobertura: requeriría un experimento de simulación adicional.

In [ ]:
summary=pd.DataFrame([dict(escenario=sc,parametros=len(NAMES[sc]),chi2=fits[sc][1],AIC_sin_constante=fits[sc][1]+2*len(NAMES[sc])) for sc in (1,2)])
summary['delta_AIC']=summary.AIC_sin_constante-summary.AIC_sin_constante.min()
summary.to_csv(OUT/'comparacion_escenarios.csv',index=False);display(summary)
print('Mejora del error al separar difusiones:',fits[1][1]-fits[2][1])
